In [5]:
import os
import glob
import numpy as np
import pandas as pd
import nibabel as nib
from scipy import ndimage

# =========================================================
# PATHS
# =========================================================

UNEST_ROOT = "/Users/abhinavkrishna/githubzz/Brain-O-jello/download_file"
SYNTH_ROOT = "/Users/abhinavkrishna/githubzz/Brain-O-jello/IXI_SYNTHSEG_BATCH"

OUT = "/Users/abhinavkrishna/githubzz/Brain-O-jello/UNesT_SynthSeg_spatial_statistics.csv"


# =========================================================
# SYNTHSEG LABEL -> CANONICAL
# =========================================================

synth_map = {
    2:  ("BRN-CER-WM", "LEFT"),
    4:  ("BRN-LAT-V", "LEFT"),
    5:  ("BRN-INF-LAT-V", "LEFT"),
    7:  ("BRN-CB-WM", "LEFT"),
    8:  ("BRN-CB-EXT", "LEFT"),
    10: ("BRN-THA", "LEFT"),
    11: ("BRN-CAU", "LEFT"),
    12: ("BRN-PUT", "LEFT"),
    13: ("BRN-PAL", "LEFT"),
    14: ("BRN-3RD-VENTRICLE", "NONE"),
    15: ("BRN-4TH-VENTRICLE", "NONE"),
    16: ("BRN-BST", "NONE"),
    17: ("BRN-HIP", "LEFT"),
    18: ("BRN-AMY", "LEFT"),
    26: ("BRN-ACC", "LEFT"),
    28: ("BRN-VDC", "LEFT"),

    41: ("BRN-CER-WM", "RIGHT"),
    43: ("BRN-LAT-V", "RIGHT"),
    44: ("BRN-INF-LAT-V", "RIGHT"),
    46: ("BRN-CB-WM", "RIGHT"),
    47: ("BRN-CB-EXT", "RIGHT"),
    49: ("BRN-THA", "RIGHT"),
    50: ("BRN-CAU", "RIGHT"),
    51: ("BRN-PUT", "RIGHT"),
    52: ("BRN-PAL", "RIGHT"),
    53: ("BRN-HIP", "RIGHT"),
    54: ("BRN-AMY", "RIGHT"),
    58: ("BRN-ACC", "RIGHT"),
    60: ("BRN-VDC", "RIGHT"),
}


canonical_to_synth = {
    v: k for k, v in synth_map.items()
}


# =========================================================
# METRICS
# =========================================================

def dice(a, b):
    a = a.astype(bool)
    b = b.astype(bool)

    inter = np.logical_and(a, b).sum()
    total = a.sum() + b.sum()

    if total == 0:
        return 1.0

    return 2.0 * inter / total


def iou(a, b):
    a = a.astype(bool)
    b = b.astype(bool)

    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()

    if union == 0:
        return 1.0

    return inter / union


def surface_distances(a, b, spacing=(1, 1, 1)):

    a = a.astype(bool)
    b = b.astype(bool)

    if not a.any() or not b.any():
        return None, None

    structure = ndimage.generate_binary_structure(3, 1)

    a_surface = a ^ ndimage.binary_erosion(
        a,
        structure=structure,
        border_value=0
    )

    b_surface = b ^ ndimage.binary_erosion(
        b,
        structure=structure,
        border_value=0
    )

    dt_a = ndimage.distance_transform_edt(
        ~a_surface,
        sampling=spacing
    )

    dt_b = ndimage.distance_transform_edt(
        ~b_surface,
        sampling=spacing
    )

    d_ab = dt_b[a_surface]
    d_ba = dt_a[b_surface]

    return d_ab, d_ba


def hd95(a, b, spacing=(1, 1, 1)):

    d_ab, d_ba = surface_distances(
        a, b, spacing
    )

    if d_ab is None:
        return np.nan

    return max(
        np.percentile(d_ab, 95),
        np.percentile(d_ba, 95)
    )


def assd(a, b, spacing=(1, 1, 1)):

    d_ab, d_ba = surface_distances(
        a, b, spacing
    )

    if d_ab is None:
        return np.nan

    return (
        d_ab.sum() + d_ba.sum()
    ) / (
        len(d_ab) + len(d_ba)
    )



# =========================================================
# FIND SUBJECTS
# =========================================================

unest_files = glob.glob(
    os.path.join(
        UNEST_ROOT,
        "*",
        "MNI305",
        "*_UNEST_segmentation_MNI305.nii.gz"
    )
)

synth_files = glob.glob(
    os.path.join(
        SYNTH_ROOT,
        "*",
        "MNI305",
        "*_SynthSeg_segmentation_MNI305.nii.gz"
    )
)

unest_dict = {
    os.path.basename(f).split("_")[0]: f
    for f in unest_files
}

synth_dict = {
    os.path.basename(f).split("_")[0]: f
    for f in synth_files
}

subjects = sorted(
    set(unest_dict) & set(synth_dict)
)

print("UNesT subjects:", len(unest_dict))
print("SynthSeg subjects:", len(synth_dict))
print("Common subjects:", len(subjects))


# =========================================================
# CALCULATE SPATIAL METRICS
# =========================================================

results = []

for i, subject_id in enumerate(subjects, 1):

    print(
        f"[{i}/{len(subjects)}] {subject_id}"
    )

    unest_img = nib.load(
        unest_dict[subject_id]
    )

    synth_img = nib.load(
        synth_dict[subject_id]
    )

    unest = np.asarray(
        unest_img.dataobj
    ).astype(np.int16)

    synth = np.asarray(
        synth_img.dataobj
    ).astype(np.int16)

    # -----------------------------------------------------
    # Verify common grid
    # -----------------------------------------------------

    if unest.shape != synth.shape:
        print(
            "  SKIP: shape mismatch",
            unest.shape,
            synth.shape
        )
        continue

    spacing = tuple(
        unest_img.header.get_zooms()[:3]
    )

    # -----------------------------------------------------
    # Each canonical structure
    # -----------------------------------------------------

    for (canonical_id, laterality), synth_label in canonical_to_synth.items():

        # UNesT source IDs for this canonical concept
        # -------------------------------------------------
        # Mapping is inferred from canonical mapping CSV
        # using the existing UNesT source mappings.

        # Load only once
        pass

UNesT subjects: 50
SynthSeg subjects: 50
Common subjects: 50
[1/50] IXI305
[2/50] IXI306
[3/50] IXI307
[4/50] IXI308
[5/50] IXI309
[6/50] IXI310
[7/50] IXI311
[8/50] IXI312
[9/50] IXI313
[10/50] IXI314
[11/50] IXI315
[12/50] IXI316
[13/50] IXI317
[14/50] IXI318
[15/50] IXI319
[16/50] IXI320
[17/50] IXI321
[18/50] IXI322
[19/50] IXI324
[20/50] IXI325
[21/50] IXI326
[22/50] IXI327
[23/50] IXI328
[24/50] IXI329
[25/50] IXI330
[26/50] IXI331
[27/50] IXI332
[28/50] IXI333
[29/50] IXI334
[30/50] IXI335
[31/50] IXI336
[32/50] IXI337
[33/50] IXI338
[34/50] IXI340
[35/50] IXI341
[36/50] IXI342
[37/50] IXI344
[38/50] IXI345
[39/50] IXI347
[40/50] IXI348
[41/50] IXI350
[42/50] IXI351
[43/50] IXI353
[44/50] IXI354
[45/50] IXI356
[46/50] IXI357
[47/50] IXI358
[48/50] IXI359
[49/50] IXI360
[50/50] IXI361


In [6]:
# =========================================================
# LOAD UNEST MAPPING
# =========================================================

MAPPING_CSV = ("/Users/abhinavkrishna/githubzz/Brain-O-jello/source_mapping_MONAI_UNEST_v0.2.7_FINAL.csv")

mapping = pd.read_csv(MAPPING_CSV)

mapping["source_id"] = (
    mapping["source_id"]
    .astype(int)
)

unest_groups = (
    mapping[
        mapping["canonical_id"].isin(
            [x[0] for x in canonical_to_synth.keys()]
        )
    ]
    .groupby(
        ["canonical_id", "laterality"]
    )["source_id"]
    .apply(list)
    .to_dict()
)

In [7]:
# =========================================================
# CALCULATE SPATIAL METRICS
# =========================================================

results = []
for i, subject_id in enumerate(subjects, 1):

    print(f"[{i}/{len(subjects)}] {subject_id}")

    unest_img = nib.load(unest_dict[subject_id])
    synth_img = nib.load(synth_dict[subject_id])

    unest = np.asarray(
        unest_img.dataobj
    ).astype(np.int16)

    synth = np.asarray(
        synth_img.dataobj
    ).astype(np.int16)

    if unest.shape != synth.shape:
        print("  SKIP shape mismatch")
        continue

    spacing = tuple(
        unest_img.header.get_zooms()[:3]
    )

    for (canonical_id, laterality), synth_label in canonical_to_synth.items():

        unest_labels = unest_groups.get(
            (canonical_id, laterality),
            []
        )

        if not unest_labels:
            continue

        # -------------------------------------------------
        # Build canonical masks
        # -------------------------------------------------

        unest_mask = np.isin(
            unest,
            unest_labels
        )

        synth_mask = (
            synth == synth_label
        )

        # -------------------------------------------------
        # Metrics
        # -------------------------------------------------

        d = dice(
            unest_mask,
            synth_mask
        )

        j = iou(
            unest_mask,
            synth_mask
        )

        h = hd95(
            unest_mask,
            synth_mask,
            spacing
        )

        a = assd(
            unest_mask,
            synth_mask,
            spacing
        )

        results.append({
            "subject_id": subject_id,
            "canonical_id": canonical_id,
            "laterality": laterality,
            "Dice": d,
            "IoU": j,
            "HD95_mm": h,
            "ASSD_mm": a,
            "UNesT_voxels": int(unest_mask.sum()),
            "SynthSeg_voxels": int(synth_mask.sum())
        })


spatial_df = pd.DataFrame(results)

spatial_df.to_csv(
    OUT,
    index=False
)

print("\nSaved:")
print(OUT)

print("\nRows:", len(spatial_df))
print("Subjects:", spatial_df.subject_id.nunique())
print(
    "Structures:",
    spatial_df[
        ["canonical_id", "laterality"]
    ].drop_duplicates().shape[0]
)

display(spatial_df.head())

[1/50] IXI305
[2/50] IXI306
[3/50] IXI307
[4/50] IXI308
[5/50] IXI309
[6/50] IXI310
[7/50] IXI311
[8/50] IXI312
[9/50] IXI313
[10/50] IXI314
[11/50] IXI315
[12/50] IXI316
[13/50] IXI317
[14/50] IXI318
[15/50] IXI319
[16/50] IXI320
[17/50] IXI321
[18/50] IXI322
[19/50] IXI324
[20/50] IXI325
[21/50] IXI326
[22/50] IXI327
[23/50] IXI328
[24/50] IXI329
[25/50] IXI330
[26/50] IXI331
[27/50] IXI332
[28/50] IXI333
[29/50] IXI334
[30/50] IXI335
[31/50] IXI336
[32/50] IXI337
[33/50] IXI338
[34/50] IXI340
[35/50] IXI341
[36/50] IXI342
[37/50] IXI344
[38/50] IXI345
[39/50] IXI347
[40/50] IXI348
[41/50] IXI350
[42/50] IXI351
[43/50] IXI353
[44/50] IXI354
[45/50] IXI356
[46/50] IXI357
[47/50] IXI358
[48/50] IXI359
[49/50] IXI360
[50/50] IXI361

Saved:
/Users/abhinavkrishna/githubzz/Brain-O-jello/UNesT_SynthSeg_spatial_statistics.csv

Rows: 1450
Subjects: 50
Structures: 29


,subject_id,canonical_id,laterality,Dice,IoU,HD95_mm,ASSD_mm,UNesT_voxels,SynthSeg_voxels
0,IXI305,BRN-CER-WM,LEFT,0.819544,0.694260,5.385165,1.439247,245679,261120
1,IXI305,BRN-LAT-V,LEFT,0.916703,0.846216,1.000000,0.552511,17238,17037
2,IXI305,BRN-INF-LAT-V,LEFT,0.467559,0.305108,3.316625,1.103266,246,725
3,IXI305,BRN-CB-WM,LEFT,0.628127,0.457861,9.110434,2.717908,8835,17666
4,IXI305,BRN-CB-EXT,LEFT,0.792393,0.656167,5.744563,2.109388,63870,63375


In [8]:
summary = (
    spatial_df
    .groupby(
        ["canonical_id", "laterality"]
    )
    .agg(
        n=("subject_id", "count"),

        Dice_mean=("Dice", "mean"),
        Dice_sd=("Dice", "std"),

        IoU_mean=("IoU", "mean"),
        IoU_sd=("IoU", "std"),

        HD95_mean_mm=("HD95_mm", "mean"),
        HD95_sd_mm=("HD95_mm", "std"),

        ASSD_mean_mm=("ASSD_mm", "mean"),
        ASSD_sd_mm=("ASSD_mm", "std"),
    )
    .reset_index()
)

display(
    summary.sort_values(
        "Dice_mean",
        ascending=False
    )
)

,canonical_id,laterality,n,Dice_mean,Dice_sd,IoU_mean,IoU_sd,HD95_mean_mm,HD95_sd_mm,ASSD_mean_mm,ASSD_sd_mm
19,BRN-LAT-V,LEFT,50,0.878059,0.126794,0.796487,0.127087,5.756089,17.011146,1.197448,2.251310
20,BRN-LAT-V,RIGHT,50,0.871792,0.131688,0.787781,0.133986,4.550992,16.930329,1.208663,2.813643
25,BRN-THA,LEFT,50,0.853343,0.133727,0.758955,0.131463,4.918087,14.405824,1.413845,2.250337
26,BRN-THA,RIGHT,50,0.849956,0.132448,0.753347,0.128699,2.960430,3.906271,1.275286,2.180899
14,BRN-CER-WM,RIGHT,50,0.821588,0.087723,0.704909,0.106835,7.775786,10.569361,1.543170,1.304814
13,BRN-CER-WM,LEFT,50,0.813142,0.087439,0.692533,0.103599,8.133117,11.274968,1.650376,1.622239
6,BRN-BST,NONE,50,0.812633,0.128871,0.699039,0.139434,15.965180,31.569970,2.302033,2.175547
22,BRN-PAL,RIGHT,50,0.801521,0.136348,0.683173,0.131219,2.895751,4.665175,1.316554,3.632312
24,BRN-PUT,RIGHT,50,0.796517,0.132214,0.675367,0.128066,3.255899,5.254221,1.402579,3.140223
9,BRN-CB-EXT,LEFT,50,0.792870,0.087119,0.663529,0.096027,63.410715,58.023464,9.897048,9.940579


In [9]:
print("=== OVERALL SPATIAL AGREEMENT ===")

for metric in [
    "Dice",
    "IoU",
    "HD95_mm",
    "ASSD_mm"
]:
    print(
        f"{metric}: "
        f"mean={spatial_df[metric].mean():.4f}, "
        f"median={spatial_df[metric].median():.4f}, "
        f"SD={spatial_df[metric].std():.4f}"
    )

=== OVERALL SPATIAL AGREEMENT ===
Dice: mean=0.7252, median=0.7920, SD=0.1864
IoU: mean=0.5962, median=0.6556, SD=0.1898
HD95_mm: mean=12.0773, median=3.3166, SD=28.9031
ASSD_mm: mean=2.5948, median=1.1399, SD=5.4829


In [10]:
# Overall
print("=== OVERALL ===")
print(spatial_df[["Dice", "IoU", "HD95_mm", "ASSD_mm"]].describe())

# Per structure
summary = (
    spatial_df
    .groupby(["canonical_id", "laterality"])
    .agg(
        n=("subject_id", "count"),
        Dice_mean=("Dice", "mean"),
        Dice_sd=("Dice", "std"),
        IoU_mean=("IoU", "mean"),
        IoU_sd=("IoU", "std"),
        HD95_mean_mm=("HD95_mm", "mean"),
        HD95_sd_mm=("HD95_mm", "std"),
        ASSD_mean_mm=("ASSD_mm", "mean"),
        ASSD_sd_mm=("ASSD_mm", "std"),
    )
    .reset_index()
)

display(summary.sort_values("Dice_mean", ascending=False))

=== OVERALL ===
              Dice          IoU      HD95_mm      ASSD_mm
count  1450.000000  1450.000000  1450.000000  1450.000000
mean      0.725220     0.596152    12.077337     2.594785
std       0.186366     0.189841    28.903082     5.482852
min       0.000000     0.000000     1.000000     0.423804
25%       0.660530     0.493127     2.236068     0.824256
50%       0.791994     0.655621     3.316625     1.139911
75%       0.840222     0.724469     6.403124     1.794334
max       0.942483     0.891222   164.950599    79.745472


,canonical_id,laterality,n,Dice_mean,Dice_sd,IoU_mean,IoU_sd,HD95_mean_mm,HD95_sd_mm,ASSD_mean_mm,ASSD_sd_mm
19,BRN-LAT-V,LEFT,50,0.878059,0.126794,0.796487,0.127087,5.756089,17.011146,1.197448,2.251310
20,BRN-LAT-V,RIGHT,50,0.871792,0.131688,0.787781,0.133986,4.550992,16.930329,1.208663,2.813643
25,BRN-THA,LEFT,50,0.853343,0.133727,0.758955,0.131463,4.918087,14.405824,1.413845,2.250337
26,BRN-THA,RIGHT,50,0.849956,0.132448,0.753347,0.128699,2.960430,3.906271,1.275286,2.180899
14,BRN-CER-WM,RIGHT,50,0.821588,0.087723,0.704909,0.106835,7.775786,10.569361,1.543170,1.304814
13,BRN-CER-WM,LEFT,50,0.813142,0.087439,0.692533,0.103599,8.133117,11.274968,1.650376,1.622239
6,BRN-BST,NONE,50,0.812633,0.128871,0.699039,0.139434,15.965180,31.569970,2.302033,2.175547
22,BRN-PAL,RIGHT,50,0.801521,0.136348,0.683173,0.131219,2.895751,4.665175,1.316554,3.632312
24,BRN-PUT,RIGHT,50,0.796517,0.132214,0.675367,0.128066,3.255899,5.254221,1.402579,3.140223
9,BRN-CB-EXT,LEFT,50,0.792870,0.087119,0.663529,0.096027,63.410715,58.023464,9.897048,9.940579


In [12]:
summary.to_csv(
    "./UNesT_SynthSeg_spatial_summary.csv",
    index=False
)

print("Saved.")

Saved.


In [13]:
print("Subjects:", spatial_df.subject_id.nunique())
print(
    "Structure/laterality pairs:",
    spatial_df[["canonical_id", "laterality"]]
    .drop_duplicates()
    .shape[0]
)

print("\nMissing metrics:")
print(spatial_df[["Dice", "IoU", "HD95_mm", "ASSD_mm"]].isna().sum())

Subjects: 50
Structure/laterality pairs: 29

Missing metrics:
Dice       0
IoU        0
HD95_mm    0
ASSD_mm    0
dtype: int64
